# 06 — DeiT Pre-trained vs Fine-tuned: 100-per-class Comparison
**Group 1 – Invoices | Step 6**

Direct comparison between:
- **DeiT-small pre-trained** (zero-shot, nessun fine-tuning su RVL-CDIP)
- **DeiT-small fine-tuned** (checkpoint salvato nel notebook 03)

Same test set: 100 examples/class × 16 classes = 1,600 total (balanced).


## 0 — Environment setup

In [ ]:
import os

def _is_kaggle():
    return os.path.exists('/kaggle/working')
def _is_colab():
    try:
        import google.colab
        return not _is_kaggle()
    except ImportError:
        return False
PLATFORM = 'kaggle' if _is_kaggle() else ('colab' if _is_colab() else 'local')
print(PLATFORM)


In [ ]:
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/NLP_Invoices/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
if PLATFORM == 'kaggle':
    OUTPUT_DIR = '/kaggle/working/outputs'
    os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
if PLATFORM == 'local':
    OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1 — Install dependencies

In [ ]:
import sys
!{sys.executable} -m pip install -q datasets transformers torchvision tqdm
print('✅ Dependencies installed')


## 1 — Imports & configuration

In [ ]:
import random, warnings, pickle
from collections import defaultdict, Counter
from io import BytesIO
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score
)
from tqdm.notebook import tqdm

# ── Configuration ────────────────────────────────────────────────────────────────────
MODEL_ID        = "facebook/deit-small-distilled-patch16-224"
N_PER_CLASS     = 100          # esempi per classe nel test bilanciato
INVOICE_LABEL   = 6
RANDOM_SEED     = 42
BATCH_SIZE      = 64
NUM_WORKERS     = 2
IMG_SIZE        = 224

# Percorso del checkpoint fine-tuned (salvato dal notebook 03)
FINETUNED_PATH  = os.path.join(OUTPUT_DIR, "deit_small_best.pt")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

LABEL_NAMES = {
    0:"advertisement", 1:"budget",   2:"email",         3:"file folder",
    4:"form",          5:"handwritten", 6:"invoice",     7:"letter",
    8:"memo",          9:"news article", 10:"presentation", 11:"questionnaire",
    12:"resume",       13:"scientific publication", 14:"scientific report",
    15:"specification"
}
ALL_LABELS    = list(LABEL_NAMES.keys())
CLASS_NAMES   = [LABEL_NAMES[i] for i in ALL_LABELS]
NUM_CLASSES   = len(CLASS_NAMES)

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
print("Config ready.")


## 2 — Load balanced test set (100 images/class)

In [ ]:
def to_pil(image_field):
    if isinstance(image_field, Image.Image):
        return image_field.convert("RGB")
    elif isinstance(image_field, dict):
        return Image.open(BytesIO(image_field["bytes"])).convert("RGB")
    else:
        return Image.fromarray(np.array(image_field)).convert("RGB")

# ── Prova prima dalla cache esistente ─────────────────────────────────────────
CACHE_DIR  = Path(OUTPUT_DIR) / "dataset_cache"
FULL_CACHE = CACHE_DIR / f"rvl_cdip_train2000_val100_testFULL_seed{RANDOM_SEED}.pkl"

test_100_examples = None

if FULL_CACHE.exists():
    print(f"Cache trovata: {FULL_CACHE.name}")
    with open(FULL_CACHE, "rb") as f:
        cached = pickle.load(f)
    full_test = cached["test"]

    # Sample N_PER_CLASS per class from cache (stratified)
    bucket = defaultdict(list)
    for ex in full_test:
        bucket[ex["label"]].append(ex)

    test_100_examples = []
    for lbl in ALL_LABELS:
        exs = bucket[lbl]
        random.shuffle(exs)
        test_100_examples.extend(exs[:N_PER_CLASS])

    random.shuffle(test_100_examples)

    missing = [LABEL_NAMES[l] for l in ALL_LABELS if len(bucket[l]) < N_PER_CLASS]
    if missing:
        print(f"  ⚠️  Classi con meno di {N_PER_CLASS} esempi nella cache: {missing}")
        print(f"  → Verranno usati tutti gli esempi disponibili per quelle classi")
else:
    print("Cache non trovata — scarico via streaming...")

# ── If the cache doesn't have enough examples, supplement from HuggingFace ──
if test_100_examples is None or any(
    sum(1 for ex in test_100_examples if ex["label"] == l) < N_PER_CLASS
    for l in ALL_LABELS
):
    if test_100_examples is not None:
        print("Integro classi mancanti da HuggingFace...")
    bucket_hf = defaultdict(list)
    # Prima copia quello che abbiamo dalla cache
    if test_100_examples:
        for ex in test_100_examples:
            if len(bucket_hf[ex["label"]]) < N_PER_CLASS:
                bucket_hf[ex["label"]].append(ex)

    need = [l for l in ALL_LABELS if len(bucket_hf[l]) < N_PER_CLASS]
    if need:
        stream = load_dataset("chainyo/rvl-cdip", split="test", streaming=True
                  ).shuffle(seed=RANDOM_SEED + 10, buffer_size=20_000)
        done_hf = set()
        for ex in stream:
            lbl = int(ex["label"])
            if lbl not in need or lbl in done_hf:
                continue
            img = to_pil(ex["image"])
            buf = BytesIO()
            img.save(buf, format="PNG")
            bucket_hf[lbl].append({"image_bytes": buf.getvalue(), "label": lbl})
            if len(bucket_hf[lbl]) >= N_PER_CLASS:
                done_hf.add(lbl)
                print(f"  ✓ {LABEL_NAMES[lbl]}: {N_PER_CLASS} esempi")
            if done_hf == set(need):
                break

        # Supplement from alternative dataset for any still-missing classes
        still_missing = [l for l in ALL_LABELS if len(bucket_hf[l]) < N_PER_CLASS]
        if still_missing:
            print(f"Integro {[LABEL_NAMES[l] for l in still_missing]} da vaclavpechtor/rvl_cdip-small-200...")
            NAME_TO_LABEL = {v: k for k, v in LABEL_NAMES.items()}
            fill_stream = load_dataset("vaclavpechtor/rvl_cdip-small-200", split="train", streaming=True)
            ext_names = fill_stream.features["label"].names
            EXT_TO_MINE = {i: NAME_TO_LABEL.get(n) for i, n in enumerate(ext_names)}
            fill_done = set()
            for ex in fill_stream:
                my_lbl = EXT_TO_MINE.get(int(ex["label"]))
                if my_lbl is None or my_lbl not in still_missing or my_lbl in fill_done:
                    continue
                img = to_pil(ex["image"])
                buf = BytesIO()
                img.save(buf, format="PNG")
                bucket_hf[my_lbl].append({"image_bytes": buf.getvalue(), "label": my_lbl})
                if len(bucket_hf[my_lbl]) >= N_PER_CLASS:
                    fill_done.add(my_lbl)
                    print(f"  ✓ {LABEL_NAMES[my_lbl]}: {N_PER_CLASS} esempi")
                if fill_done == set(still_missing):
                    break

    test_100_examples = [ex for exs in bucket_hf.values() for ex in exs[:N_PER_CLASS]]
    random.shuffle(test_100_examples)

true_labels = [ex["label"] for ex in test_100_examples]
dist = Counter(true_labels)
print(f"\n✅ Test bilanciato: {len(test_100_examples)} esempi")
for lbl in ALL_LABELS:
    print(f"  {LABEL_NAMES[lbl]:<30} {dist.get(lbl,0):>4}")


## 3 — Dataset & DataLoader

In [ ]:
class DocDataset(Dataset):
    def __init__(self, examples, processor):
        self.examples  = examples
        self.processor = processor

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        ex = self.examples[i]
        try:
            img = Image.open(BytesIO(ex["image_bytes"])).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), 0)
        inputs = self.processor(images=img, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), int(ex["label"])


def make_loader(examples, processor):
    ds = DocDataset(examples, processor)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
print("DataLoader ready.")


## 4 — Inferenza: DeiT pre-trained (zero-shot)
The original model was not trained on RVL-CDIP.  
Its 1000 ImageNet classes do not correspond to the 16 document classes —  
classification is performed by name-matching the top logit, falling back to random.  
Ci aspettiamo performance molto basse, specialmente su invoice.


In [ ]:
# Load the original DeiT model (1000 ImageNet classes)
model_pretrained = AutoModelForImageClassification.from_pretrained(MODEL_ID)
model_pretrained = model_pretrained.to(DEVICE)
model_pretrained.eval()
print(f"DeiT pre-trained caricato ({sum(p.numel() for p in model_pretrained.parameters())/1e6:.1f}M params)")

loader = make_loader(test_100_examples, processor)
preds_pretrained = []

with torch.no_grad():
    for pixel_values, _ in tqdm(loader, desc="Pre-trained inference"):
        pixel_values = pixel_values.to(DEVICE)
        logits = model_pretrained(pixel_values=pixel_values).logits
        # Prende il top-1 e lo mappa modulo NUM_CLASSES
        # (heuristic: 1000 ImageNet classes have no direct mapping to document classes)
        top_idx = logits.argmax(dim=-1).cpu()
        mapped  = (top_idx % NUM_CLASSES).tolist()
        preds_pretrained.extend(mapped)

acc_pre      = accuracy_score(true_labels, preds_pretrained)
f1_pre       = f1_score(true_labels, preds_pretrained, average="macro", zero_division=0)
inv_true_bin = [1 if l == INVOICE_LABEL else 0 for l in true_labels]
inv_pred_bin = [1 if p == INVOICE_LABEL else 0 for p in preds_pretrained]
inv_f1_pre   = f1_score(inv_true_bin, inv_pred_bin, zero_division=0)

print(f"\n{'─'*55}")
print(f"  DeiT-small PRE-TRAINED (zero-shot)")
print(f"{'─'*55}")
print(f"  Overall Accuracy : {acc_pre:.3f}")
print(f"  Macro F1         : {f1_pre:.3f}")
print(f"  Invoice F1       : {inv_f1_pre:.3f}\n")
print(classification_report(true_labels, preds_pretrained, target_names=CLASS_NAMES,
                             labels=ALL_LABELS, digits=3, zero_division=0))
del model_pretrained; torch.cuda.empty_cache()


## 5 — Inference: DeiT fine-tuned (checkpoint from notebook 03)

In [ ]:
if not os.path.isfile(FINETUNED_PATH):
    raise FileNotFoundError(
        f"Checkpoint non trovato: {FINETUNED_PATH}\n"
        "Assicurati di aver eseguito il notebook 03 prima di questo."
    )

model_finetuned = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    id2label={i: n for i, n in LABEL_NAMES.items()},
    label2id={n: i for i, n in LABEL_NAMES.items()},
)
model_finetuned.load_state_dict(torch.load(FINETUNED_PATH, map_location=DEVICE))
model_finetuned = model_finetuned.to(DEVICE)
model_finetuned.eval()
print(f"✅ DeiT fine-tuned caricato da {FINETUNED_PATH}")

preds_finetuned = []
with torch.no_grad():
    for pixel_values, _ in tqdm(loader, desc="Fine-tuned inference"):
        pixel_values = pixel_values.to(DEVICE)
        logits = model_finetuned(pixel_values=pixel_values).logits
        preds_finetuned.extend(logits.argmax(dim=-1).cpu().tolist())

acc_ft      = accuracy_score(true_labels, preds_finetuned)
f1_ft       = f1_score(true_labels, preds_finetuned, average="macro", zero_division=0)
inv_pred_ft = [1 if p == INVOICE_LABEL else 0 for p in preds_finetuned]
inv_f1_ft   = f1_score(inv_true_bin, inv_pred_ft, zero_division=0)

print(f"\n{'─'*55}")
print(f"  DeiT-small FINE-TUNED")
print(f"{'─'*55}")
print(f"  Overall Accuracy : {acc_ft:.3f}")
print(f"  Macro F1         : {f1_ft:.3f}")
print(f"  Invoice F1       : {inv_f1_ft:.3f}\n")
print(classification_report(true_labels, preds_finetuned, target_names=CLASS_NAMES,
                             labels=ALL_LABELS, digits=3, zero_division=0))
del model_finetuned; torch.cuda.empty_cache()


## 6 — Side-by-side confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(26, 11))

for ax, preds, title in [
    (axes[0], preds_pretrained, f"Pre-trained (acc={acc_pre:.3f})"),
    (axes[1], preds_finetuned,  f"Fine-tuned  (acc={acc_ft:.3f})"),
]:
    cm = confusion_matrix(true_labels, preds, labels=ALL_LABELS)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, colorbar=False, cmap="Blues", xticks_rotation=45)
    ax.set_title(f"DeiT-small — {title}", fontsize=11, pad=12)

plt.suptitle("Pre-trained vs Fine-tuned — RVL-CDIP 100/classe", fontsize=13, y=1.01)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "06_confusion_matrices_comparison.png")
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {path}")


## 7 — Per-class F1 bar chart

In [ ]:
# Per-class F1 for both models
from sklearn.metrics import f1_score

f1_per_class_pre = f1_score(true_labels, preds_pretrained, average=None,
                             labels=ALL_LABELS, zero_division=0)
f1_per_class_ft  = f1_score(true_labels, preds_finetuned,  average=None,
                             labels=ALL_LABELS, zero_division=0)

x   = np.arange(NUM_CLASSES)
w   = 0.38
fig, ax = plt.subplots(figsize=(18, 6))
b1 = ax.bar(x - w/2, f1_per_class_pre, w, label="Pre-trained", color="#95a5a6", alpha=0.85)
b2 = ax.bar(x + w/2, f1_per_class_ft,  w, label="Fine-tuned",  color="#2980b9", alpha=0.85)

# Evidenzia invoice
inv_idx = ALL_LABELS.index(INVOICE_LABEL)
b1[inv_idx].set_color("#e74c3c"); b1[inv_idx].set_alpha(0.9)
b2[inv_idx].set_color("#27ae60"); b2[inv_idx].set_alpha(0.9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.15)
ax.set_title("F1 per classe — Pre-trained vs Fine-tuned (rosso=invoice pre, verde=invoice ft)", fontsize=11)
ax.legend(fontsize=10)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "06_f1_per_class_comparison.png")
fig.savefig(path, dpi=150)
plt.show()
print(f"Saved → {path}")


## 8 — Delta F1: improvement from fine-tuning

In [ ]:
delta = f1_per_class_ft - f1_per_class_pre
colors = ["#27ae60" if d >= 0 else "#e74c3c" for d in delta]
colors[inv_idx] = "#8e44ad"  # viola per invoice

fig, ax = plt.subplots(figsize=(18, 5))
bars = ax.bar(x, delta, color=colors, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Δ F1 (fine-tuned − pre-trained)")
ax.set_title("Guadagno F1 per classe dopo fine-tuning (viola = invoice)", fontsize=11)

for bar, d in zip(bars, delta):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (0.01 if d >= 0 else -0.04),
            f"{d:+.2f}", ha="center", fontsize=7.5)

ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR, "06_delta_f1_per_class.png")
fig.savefig(path, dpi=150)
plt.show()
print(f"Saved → {path}")

# Classes that regressed after fine-tuning
losers = [(LABEL_NAMES[ALL_LABELS[i]], delta[i]) for i in range(NUM_CLASSES) if delta[i] < 0]
if losers:
    print("\n⚠️  Classi che hanno perso F1 dopo fine-tuning:")
    for name, d in sorted(losers, key=lambda x: x[1]):
        print(f"  {name:<30} {d:+.3f}")


## 9 — Summary table

In [ ]:
print("\n── Summary (100 esempi/classe, test bilanciato) ──────────────────")
print(f"{'Metrica':<20} {'Pre-trained':>14} {'Fine-tuned':>12} {'Delta':>8}")
print("─" * 57)

for metrica, v_pre, v_ft in [
    ("Overall Accuracy", acc_pre,    acc_ft),
    ("Macro F1",         f1_pre,     f1_ft),
    ("Invoice F1",       inv_f1_pre, inv_f1_ft),
]:
    d = v_ft - v_pre
    tag = "▲" if d > 0 else ("▼" if d < 0 else "─")
    print(f"{metrica:<20} {v_pre:>14.3f} {v_ft:>12.3f} {tag}{abs(d):>6.3f}")

print()
print(f"{'Classe':<30} {'Pre F1':>8} {'FT F1':>8} {'Delta':>8}")
print("─" * 57)
for i, lbl in enumerate(ALL_LABELS):
    d    = f1_per_class_ft[i] - f1_per_class_pre[i]
    tag  = "▲" if d > 0 else ("▼" if d < 0 else "─")
    star = " ★" if lbl == INVOICE_LABEL else ""
    print(f"{LABEL_NAMES[lbl]:<30} {f1_per_class_pre[i]:>8.3f} {f1_per_class_ft[i]:>8.3f} {tag}{abs(d):>6.3f}{star}")
print("  ★ = invoice")


## 10 — Save results

In [ ]:
results_path = os.path.join(OUTPUT_DIR, "06_pretrained_vs_finetuned_results.txt")
with open(results_path, "w") as f:
    f.write("DeiT-small: Pre-trained vs Fine-tuned — Confronto\n")
    f.write("=" * 55 + "\n\n")
    f.write(f"Test set    : {N_PER_CLASS}/classe x {NUM_CLASSES} = {len(test_100_examples)} esempi\n")
    f.write(f"Checkpoint  : {FINETUNED_PATH}\n\n")
    f.write(f"{'Metrica':<20} {'Pre-trained':>14} {'Fine-tuned':>12} {'Delta':>8}\n")
    f.write("─" * 57 + "\n")
    for metrica, v_pre, v_ft in [
        ("Overall Accuracy", acc_pre,    acc_ft),
        ("Macro F1",         f1_pre,     f1_ft),
        ("Invoice F1",       inv_f1_pre, inv_f1_ft),
    ]:
        d = v_ft - v_pre
        tag = "▲" if d > 0 else "▼"
        f.write(f"{metrica:<20} {v_pre:>14.3f} {v_ft:>12.3f} {tag}{abs(d):>6.3f}\n")
    f.write("\n── F1 per classe ──\n")
    for i, lbl in enumerate(ALL_LABELS):
        d = f1_per_class_ft[i] - f1_per_class_pre[i]
        tag = "▲" if d > 0 else "▼"
        f.write(f"{LABEL_NAMES[lbl]:<30} {f1_per_class_pre[i]:>8.3f} {f1_per_class_ft[i]:>8.3f} {tag}{abs(d):>6.3f}\n")

print(f"Results saved → {results_path}")
print("\n✅ Step 6 (confronto pre-trained vs fine-tuned) complete.")
